# Physics-Informed PPO Relaxation

Train a PPO agent to relax a mesh by moving vertices, reaching or approaching the
energy minimised by bilevel optimisation — but in far fewer steps.

## Architecture
- **GNN**: MeshGraphNet (edge→node message passing, no equivariance constraint).
- **Action**: per-vertex displacement (factorised Gaussian policy), mean guided by `-∇E`.
- **Action projection**: clip displacements so no edge falls below `min_dist_T1`,
  no vertex leaves the domain, and no vertex crosses an edge.
- **T1 transitions**: applied after projection iff they lower the energy.
- **Angles**: updated by gradient descent each step (like the bilevel inner loop).

## Reward
Normalised energy reduction, plus shape terms (area dispersion, min-area, inverted-area)
to keep cells balanced and non-degenerate. The energy is exactly `energy_adaptive`,
the same function and parameters as the bilevel baseline (verified in cell 1).

## Pipeline (run cells in order)
1. **Setup** — build mesh, bilevel baseline, static arrays, energy-consistency check.
2. **MeshGraphNet** — policy/value network.
3. **Environment** — vectorised JAX env (256 parallel envs).
4. **PPO training** — GPU-batched rollout + update, saves best model.
5. **Evaluation** — load model, compare PPO vs bilevel (curves + mesh geometry).

In [ ]:
import os, sys, math
sys.path.insert(0, '/home/mcorbeau/Bureau/vertax/src/vertax/mitotic_spindle')
os.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE', 'false')

import numpy as np
import jax
import jax.numpy as jnp
from jax import vmap, jit, grad
import optax
import matplotlib.pyplot as plt
from functools import partial
from topo_patched import update_T1_bounded_safe

from vertax import BoundedMesh
from vertax.topo import update_T1_bounded
from energy_adaptive import energy_adaptive
from boundedbop_patched import BoundedBilevelOptimizer

# ── Constants ─────────────────────────────────────────────────────────────────
N_CELLS     = 4
MIN_DIST_T1 = 0.01
MAX_STEPS   = 50
N_ENVS      = 256
ANG_MIN     = 0.017
ANG_MAX     = float(jnp.pi / 2 - 0.01)
POS_MIN     = -0.5
POS_MAX     = 2.5

# MeshGraphNet
H_DIM    = 128
N_LAYERS = 3

print('JAX devices:', jax.devices())


def mesh_energy(mesh):
    return float(energy_adaptive(
        jnp.array(mesh.vertices), jnp.array(mesh.angles), jnp.array(mesh.edges),
        jnp.array(mesh.faces), None, None, None,
        jnp.array(mesh.vertices_params), jnp.array(mesh.edges_params),
        jnp.array(mesh.faces_params),
    ))


def make_random_mesh(seed):
    mesh = BoundedMesh.from_random_seeds(
        nb_seeds=N_CELLS, width=math.sqrt(N_CELLS), height=math.sqrt(N_CELLS), random_key=seed
    )
    mesh.vertices_params = jnp.zeros(mesh.nb_vertices)
    mesh.edges_params    = jax.random.uniform(jax.random.PRNGKey(seed), (mesh.nb_edges,)) * 20 - 10
    mesh.faces_params    = jnp.full(mesh.nb_faces, 1.0)
    return mesh


# ── Build mesh and bilevel baseline ──────────────────────────────────────────
mesh_init    = make_random_mesh(seed=42)
E_init       = mesh_energy(mesh_init)
print(f'Initial mesh — {mesh_init.nb_faces} cells, E={E_init:.4f}')

mesh_bilevel = BoundedMesh.copy_mesh(mesh_init)
opt = BoundedBilevelOptimizer()
opt.loss_function_inner = energy_adaptive
opt.inner_solver        = optax.adam(learning_rate=1e-4)
opt.update_T1           = True
opt._update_T1_func     = update_T1_bounded
opt.min_dist_T1         = MIN_DIST_T1
opt.max_nb_iterations   = 2000
opt.tolerance           = 1e-6
opt.patience            = 50
opt.inner_optimization(mesh_bilevel)
E_bilevel = mesh_energy(mesh_bilevel)
print(f'Bilevel baseline — E={E_bilevel:.4f}')
print(f'Target: E ≤ {E_bilevel:.4f}')

# ── Static mesh arrays ─────────────────────────────────────────────────────────
HE0   = jnp.array(mesh_init.edges)
FACE0 = jnp.array(mesh_init.faces)
VP0   = jnp.array(mesh_init.vertices_params)
EP0   = jnp.array(mesh_init.edges_params)
FP0   = jnp.array(mesh_init.faces_params)
X0    = jnp.array(mesh_init.vertices)
ANG0  = jnp.clip(jnp.array(mesh_init.angles), ANG_MIN, ANG_MAX)

N_VERTS = X0.shape[0]
N_HE    = HE0.shape[0]

print(f'Static arrays — X0={X0.shape}, HE0={HE0.shape}, n_verts={N_VERTS}, n_he={N_HE}')
print(f'Position bounds: [{POS_MIN}, {POS_MAX}], MAX_STEPS={MAX_STEPS}, N_ENVS={N_ENVS}')

# ── Sanity check: PPO uses the SAME energy parameters as the bilevel baseline ──
# PPO eval calls energy_adaptive with VP0/EP0/FP0 (frozen from mesh_init).
# The bilevel only ran inner_optimization (positions+angles), so its tension/area
# params should be unchanged. If these match, E_bilevel and E_ppo are comparable.
_vp_match = np.allclose(np.array(VP0), np.array(mesh_bilevel.vertices_params))
_ep_match = np.allclose(np.array(EP0), np.array(mesh_bilevel.edges_params))
_fp_match = np.allclose(np.array(FP0), np.array(mesh_bilevel.faces_params))
print(f'\n── Energy-parameter consistency (PPO vs bilevel) ──')
print(f'  vertices_params identical: {_vp_match}')
print(f'  edges_params    identical: {_ep_match}')
print(f'  faces_params    identical: {_fp_match}')
print(f'  → Energies are {"COMPARABLE ✓" if (_vp_match and _ep_match and _fp_match) else "NOT directly comparable ✗"}')


## 2. MeshGraphNet policy

The policy network: a **MeshGraphNet** (edge↔node message passing) that outputs, per vertex, the mean and log-std of a Gaussian over the displacement, plus a state value (critic).

**Architecture:**
- Node-feature encoder (position, energy gradient, degree) and edge-feature encoder (relative vector, distance, tension).
- `N_LAYERS` message-passing blocks with residual connections.
- Heads: `alpha_head` (per-vertex step size, via softplus), `resid_head` (learned correction), `log_std_head` (exploration), `value_head` (critic).

**Physics-guided action:** the policy mean is `mean = -grad_En · alpha + residual`, i.e. an **energy gradient-descent step** (`-grad_En`, normalised gradient) scaled by a learned step size, plus a learned residual correction. The network therefore starts close to a useful descent step from episode 1.

`ACTION_SCALE` is deliberately small (`0.003`) so steps are ~10× smaller than a typical edge length → a vertex cannot jump across an edge in one step (like the bilevel with its tiny learning rate).

In [ ]:
# ── MeshGraphNet ──────────────────────────────────────────────────────────────
#
# ACTION_SCALE = 0.003 → steps ~10× smaller than typical edge length (~0.2),
# making it geometrically impossible for a vertex to jump across an edge in one step.
# This gives the same crossing-safety guarantee the bilevel gets from its tiny lr.

NODE_DIM     = 5
EDGE_DIM     = 4
ACTION_SCALE = 0.003
ACTION_CAP   = 5.0 * ACTION_SCALE


def mlp_init(key, sizes, small_last=False):
    params = []
    for i, (n_in, n_out) in enumerate(zip(sizes[:-1], sizes[1:])):
        key, sk = jax.random.split(key)
        is_last = (i == len(sizes) - 2)
        scale   = 1e-3 if (is_last and small_last) else math.sqrt(2.0 / n_in)
        W = jax.random.normal(sk, (n_in, n_out)) * scale
        b = jnp.zeros(n_out)
        params.append((W, b))
    return params


def mlp(params, x):
    for i, (W, b) in enumerate(params):
        x = x @ W + b
        if i < len(params) - 1:
            x = jax.nn.relu(x)
    return x


def init_mgn(key, h_dim=H_DIM, n_layers=N_LAYERS):
    p = {}
    key, k = jax.random.split(key)
    p['node_enc'] = mlp_init(k, [NODE_DIM, h_dim, h_dim])
    key, k = jax.random.split(key)
    p['edge_enc'] = mlp_init(k, [EDGE_DIM, h_dim, h_dim])
    p['layers'] = []
    for _ in range(n_layers):
        key, k1, k2 = jax.random.split(key, 3)
        p['layers'].append({
            'phi_e': mlp_init(k1, [h_dim*3, h_dim, h_dim]),
            'phi_v': mlp_init(k2, [h_dim*2, h_dim, h_dim]),
        })
    key, k1, k2, k3 = jax.random.split(key, 4)
    p['alpha_head']   = mlp_init(k1, [h_dim, h_dim, 2], small_last=True)
    p['resid_head']   = mlp_init(k2, [h_dim, h_dim, 2], small_last=True)
    p['log_std_head'] = mlp_init(k3, [h_dim, h_dim, 2], small_last=True)
    key, k = jax.random.split(key)
    p['value_head'] = mlp_init(k, [h_dim, h_dim, 1])
    return p


def mgn_forward(p, X, srcs, tgts, tensions, grad_E, degree):
    n_verts = X.shape[0]
    g_norm  = jnp.linalg.norm(grad_E) + 1e-8
    grad_En = grad_E / g_norm

    node_feat = jnp.concatenate([X, grad_En, degree[:, None]], axis=1)
    H = vmap(lambda f: mlp(p['node_enc'], f))(node_feat)

    dX        = X[srcs] - X[tgts]
    dist      = jnp.linalg.norm(dX, axis=1, keepdims=True)
    edge_feat = jnp.concatenate([dX, dist, tensions[:, None]], axis=1)
    E_feat    = vmap(lambda f: mlp(p['edge_enc'], f))(edge_feat)

    for layer in p['layers']:
        msg_in = jnp.concatenate([H[srcs], H[tgts], E_feat], axis=1)
        E_feat = E_feat + vmap(lambda x: mlp(layer['phi_e'], x))(msg_in)
        agg    = jnp.zeros((n_verts, H_DIM)).at[tgts].add(E_feat)
        H = H + vmap(lambda x: mlp(layer['phi_v'], x))(jnp.concatenate([H, agg], axis=1))

    alpha    = jax.nn.softplus(vmap(lambda h: mlp(p['alpha_head'], h))(H)) * ACTION_SCALE
    residual = vmap(lambda h: mlp(p['resid_head'], h))(H)
    mean     = -grad_En * alpha + residual * ACTION_SCALE
    mean     = jnp.clip(mean, -ACTION_CAP, ACTION_CAP)
    log_std  = jnp.clip(vmap(lambda h: mlp(p['log_std_head'], h))(H), -4.0, 0.0)
    value    = jnp.mean(vmap(lambda h: mlp(p['value_head'], h))(H))
    return mean, log_std, value


# Smoke test
key_mgn    = jax.random.PRNGKey(0)
mgn_params = init_mgn(key_mgn)

X_test    = jnp.array(mesh_init.vertices)
he_np     = np.array(mesh_init.edges)
he_even   = he_np[::2]
srcs_test = he_even[:, 3] + he_even[:, 5] - 2
tgts_test = he_even[:, 4] + he_even[:, 6] - 3
valid     = (srcs_test >= 0) & (tgts_test >= 0)
srcs_test = jnp.array(srcs_test[valid]); tgts_test = jnp.array(tgts_test[valid])
tens_test = jnp.ones(int(valid.sum()))
deg_test  = jnp.ones(mesh_init.nb_vertices) * 5.0
ang_t  = jnp.array(mesh_init.angles); he_t = jnp.array(mesh_init.edges)
face_t = jnp.array(mesh_init.faces);  vp_t = jnp.array(mesh_init.vertices_params)
ep_t   = jnp.array(mesh_init.edges_params); fp_t = jnp.array(mesh_init.faces_params)
gE_test = jax.grad(lambda Xg: energy_adaptive(
    Xg, ang_t, he_t, face_t, None, None, None, vp_t, ep_t, fp_t
))(X_test)

mean_t, lgs_t, val_t = mgn_forward(mgn_params, X_test, srcs_test, tgts_test, tens_test, gE_test, deg_test)
print(f'MeshGraphNet OK — |mean| max={float(jnp.max(jnp.abs(mean_t))):.2e}  '
      f'(ACTION_SCALE={ACTION_SCALE}, edge length ~0.2 → no crossing possible)')
n_params = sum(W.size + b.size for layers in [
    mgn_params['node_enc'], mgn_params['edge_enc'], mgn_params['alpha_head'],
    mgn_params['resid_head'], mgn_params['log_std_head'], mgn_params['value_head'],
    *[v for l in mgn_params['layers'] for v in l.values()]
] for W, b in layers)
print(f'Total parameters: {n_params:,}')


## 3. Vectorised RL environment

Defines the environment dynamics, entirely in JAX (vmappable over the 256 envs).

**Key functions:**
- `build_graph_jax`: extracts `(srcs, tgts, tensions, degree, mask)` from the half-edge table in pure JAX (boundary ghost edges are masked out).
- `energy_of` / `grad_energy_of` / `grad_energy_ang`: energy and its gradients w.r.t. positions and angles.
- `cell_areas`, `area_dispersion`, `min_area`, `neg_area`: area metrics (the last one detects **inverted** cells = negative signed area).
- `project_action`: **geometric safeguard**. Bounds the action so that (1) vertices stay within `[POS_MIN, POS_MAX]`, (2) no edge falls below `min_dist_T1`, (3) no vertex **crosses** an edge (segment-segment intersection test).
- `env_step`: one full step — PPO action on `X` (guided by `-grad_E`), gradient step on the angles (like the bilevel), conditional T1 (applied only if it lowers energy).

**Reward** = normalised energy reduction − area-dispersion penalty + min-area bonus − strong inverted-area penalty. All clamped to `[-1, 1]`.

In [ ]:
# ── Vectorised JAX environment ────────────────────────────────────────────────
#
# Reward = normalised energy reduction + area shape terms.
# Angles are updated by gradient descent every step (like the bilevel inner loop),
# independently of the PPO action on vertices. This keeps the arc-edge geometry
# consistent with the vertex positions.

from vertax.geo import get_area_bounded

LOG_STD_MIN = -4.0
LOG_STD_MAX = -1.5
REWARD_CLIP = 1.0
K_AREA_VAR  = 5.0
K_AREA_MIN  = 2.0
K_NEG_AREA  = 50.0
ANG_LR      = 1e-3    # gradient step size for angles (same order as bilevel lr)


def build_graph_jax(he, ep):
    he_even = he[::2]
    src_all = he_even[:, 3] + he_even[:, 5] - 2
    tgt_all = he_even[:, 4] + he_even[:, 6] - 3
    valid   = (src_all >= 0) & (tgt_all >= 0)
    mask    = valid.astype(jnp.float32)
    srcs = jnp.where(valid, src_all, 0)
    tgts = jnp.where(valid, tgt_all, 0)
    tensions = (jax.nn.sigmoid(ep) + 1) * mask
    deg = jnp.zeros(N_VERTS).at[srcs].add(mask).at[tgts].add(mask)
    return srcs, tgts, tensions, deg, mask


def energy_of(X, ang, he, face):
    return energy_adaptive(X, ang, he, face, None, None, None, VP0, EP0, FP0)


def grad_energy_of(X, ang, he, face):
    g = jax.grad(lambda Xg: energy_of(Xg, ang, he, face))(X)
    return jnp.nan_to_num(g, nan=0.0, posinf=0.0, neginf=0.0)


def grad_energy_ang(X, ang, he, face):
    """Gradient of energy w.r.t. angles."""
    g = jax.grad(lambda a: energy_of(X, a, he, face))(ang)
    return jnp.nan_to_num(g, nan=0.0, posinf=0.0, neginf=0.0)


def cell_areas(X, ang, he, face):
    vt_full  = jnp.vstack([jnp.array([[0.0, 0.0], [1.0, 1.0]]), X])
    ang_full = jnp.repeat(ang, 2)
    faces    = jnp.arange(face.shape[0])
    return vmap(lambda f: get_area_bounded(f, vt_full, ang_full, he, face))(faces)


def area_dispersion(X, ang, he, face):
    areas  = cell_areas(X, ang, he, face)
    target = FP0[0]
    return jnp.mean(((areas - target) / (target + 1e-8)) ** 2)


def min_area(X, ang, he, face):
    areas  = cell_areas(X, ang, he, face)
    return jnp.min(areas) / (FP0[0] + 1e-8)


def neg_area(X, ang, he, face):
    areas = cell_areas(X, ang, he, face)
    return jnp.sum(jax.nn.relu(-areas)) / (FP0[0] + 1e-8)


def get_edge_lengths_masked(X, srcs, tgts, mask):
    L = jnp.linalg.norm(X[srcs] - X[tgts], axis=1)
    return jnp.where(mask > 0, L, 1e3)


def _crossing_alpha(p, dp, a, da, b, db):
    def cross2d(u, v):
        return u[0] * v[1] - u[1] * v[0]
    B  = b - a;   dB = db - da
    P  = p - a;   dP = dp - da
    c0 = cross2d(B,  P)
    c1 = cross2d(B, dP) + cross2d(dB, P)
    c2 = cross2d(dB, dP)
    eps = 1e-7
    disc      = c1 * c1 - 4.0 * c2 * c0
    disc_safe = jnp.maximum(disc, 0.0)
    alpha1 = jnp.where(jnp.abs(c2) > eps,
                       (-c1 - jnp.sqrt(disc_safe)) / (2.0 * c2 + eps),
                       jnp.where(jnp.abs(c1) > eps, -c0 / c1, 1.0))
    alpha2 = jnp.where(jnp.abs(c2) > eps,
                       (-c1 + jnp.sqrt(disc_safe)) / (2.0 * c2 + eps), 1.0)
    def on_segment(alpha):
        pt = p + alpha * dp;  at = a + alpha * da;  bt = b + alpha * db
        seg  = bt - at
        proj = jnp.dot(pt - at, seg) / (jnp.dot(seg, seg) + eps)
        return (proj >= 0.0) & (proj <= 1.0)
    def safe_alpha(alpha):
        ok = (alpha > eps) & (alpha <= 1.0) & (disc >= 0.0) & on_segment(alpha)
        return jnp.where(ok, alpha * (1.0 - eps), 1.0)
    return jnp.minimum(safe_alpha(alpha1), safe_alpha(alpha2))


def project_action(X, action, srcs, tgts, mask, min_dist=MIN_DIST_T1):
    action = jnp.clip(X + action, POS_MIN, POS_MAX) - X
    X_new       = X + action
    new_lengths = get_edge_lengths_masked(X_new, srcs, tgts, mask)
    old_lengths = get_edge_lengths_masked(X,     srcs, tgts, mask)
    violated    = new_lengths < min_dist
    safe_len    = jnp.where(
        violated,
        jnp.clip((old_lengths - min_dist) / (old_lengths - new_lengths + 1e-8), 0.0, 1.0),
        1.0
    )
    n_e = srcs.shape[0]
    def alpha_for_pair(v_idx, e_idx):
        s = srcs[e_idx]; t = tgts[e_idx]; m = mask[e_idx]
        is_endpoint = (v_idx == s) | (v_idx == t)
        alpha = _crossing_alpha(X[v_idx], action[v_idx],
                                X[s], action[s], X[t], action[t])
        return jnp.where(m > 0, jnp.where(is_endpoint, 1.0, alpha), 1.0)
    vi = jnp.arange(N_VERTS)
    ei = jnp.arange(n_e)
    cross_alphas = vmap(lambda v: vmap(lambda e: alpha_for_pair(v, e))(ei))(vi)
    alpha = jnp.minimum(jnp.min(safe_len), jnp.min(cross_alphas))
    return X + alpha * action, alpha


def mgn_forward_masked(p, X, srcs, tgts, tensions, grad_E, degree, edge_mask):
    n_verts = X.shape[0]
    g_norm  = jnp.linalg.norm(grad_E) + 1e-8
    grad_En = grad_E / g_norm
    node_feat = jnp.concatenate([X, grad_En, degree[:, None]], axis=1)
    H = vmap(lambda f: mlp(p['node_enc'], f))(node_feat)
    dX        = X[srcs] - X[tgts]
    dist      = jnp.linalg.norm(dX, axis=1, keepdims=True)
    edge_feat = jnp.concatenate([dX, dist, tensions[:, None]], axis=1)
    E_feat    = vmap(lambda f: mlp(p['edge_enc'], f))(edge_feat)
    E_feat    = E_feat * edge_mask[:, None]
    for layer in p['layers']:
        msg_in = jnp.concatenate([H[srcs], H[tgts], E_feat], axis=1)
        msg    = vmap(lambda x: mlp(layer['phi_e'], x))(msg_in) * edge_mask[:, None]
        E_feat = E_feat + msg
        agg    = jnp.zeros((n_verts, H_DIM)).at[tgts].add(E_feat)
        H = H + vmap(lambda x: mlp(layer['phi_v'], x))(jnp.concatenate([H, agg], axis=1))
    alpha    = jax.nn.softplus(vmap(lambda h: mlp(p['alpha_head'], h))(H)) * ACTION_SCALE
    residual = vmap(lambda h: mlp(p['resid_head'], h))(H)
    mean     = -grad_En * alpha + residual * ACTION_SCALE
    mean     = jnp.clip(mean, -ACTION_CAP, ACTION_CAP)
    log_std  = jnp.clip(vmap(lambda h: mlp(p['log_std_head'], h))(H), LOG_STD_MIN, LOG_STD_MAX)
    value    = jnp.mean(vmap(lambda h: mlp(p['value_head'], h))(H))
    return mean, log_std, value


@jit
def env_step(params, X, ang, he, face, key):
    """One env step: PPO action on X + gradient step on ang (like bilevel)."""
    srcs, tgts, tensions, degree, mask = build_graph_jax(he, EP0)
    grad_E = grad_energy_of(X, ang, he, face)

    mean, log_std, value = mgn_forward_masked(params, X, srcs, tgts, tensions, grad_E, degree, mask)

    noise   = jax.random.normal(key, mean.shape)
    action  = mean + jnp.exp(log_std) * noise * ACTION_SCALE
    logp    = -0.5 * jnp.sum(((action - mean) / (jnp.exp(log_std) * ACTION_SCALE + 1e-8)) ** 2
                             + 2 * (log_std + jnp.log(ACTION_SCALE)) + jnp.log(2 * jnp.pi))

    E_before  = energy_of(X, ang, he, face)
    ap_before = area_dispersion(X, ang, he, face)
    ma_before = min_area(X, ang, he, face)

    X_new, _ = project_action(X, action, srcs, tgts, mask)

    # Gradient step on angles — keeps arc geometry consistent with new positions
    g_ang = grad_energy_ang(X_new, ang, he, face)
    ang_new = jnp.clip(ang - ANG_LR * g_ang, ANG_MIN, ANG_MAX)

    vt1, at1, ht1, ft1 = update_T1_bounded_safe(
        X_new, ang_new, he, face, VP0, EP0, FP0, energy_adaptive, MIN_DIST_T1
    )
    E_no_t1 = energy_of(X_new, ang_new, he, face)
    E_t1    = energy_of(vt1, at1, ht1, ft1)
    use_t1  = E_t1 < E_no_t1
    X_out   = jnp.where(use_t1, vt1, X_new)
    ang_out = jnp.where(use_t1, at1, ang_new)
    he_out  = jnp.where(use_t1, ht1, he)
    face_out= jnp.where(use_t1, ft1, face)

    E_new    = jnp.nan_to_num(energy_of(X_out, ang_out, he_out, face_out), nan=1e12, posinf=1e12)
    ap_after = jnp.nan_to_num(area_dispersion(X_out, ang_out, he_out, face_out), nan=1e3, posinf=1e3)
    ma_after = jnp.nan_to_num(min_area(X_out, ang_out, he_out, face_out), nan=-1e3, neginf=-1e3)
    na_after = jnp.nan_to_num(neg_area(X_out, ang_out, he_out, face_out), nan=1e3, posinf=1e3)

    r_energy = (E_before - E_new) / (abs(E_init) + 1e-8)
    r_var    = -(ap_after - ap_before)
    r_min    =  (ma_after - ma_before)
    r_neg    = -na_after
    reward   = jnp.clip(r_energy + K_AREA_VAR * r_var + K_AREA_MIN * r_min + K_NEG_AREA * r_neg,
                        -REWARD_CLIP, REWARD_CLIP)

    return X_out, ang_out, he_out, face_out, action, logp, value, reward, E_new


env_step_batch = jit(vmap(env_step, in_axes=(None, 0, 0, 0, 0, 0)))


def reset_envs(n):
    X   = jnp.broadcast_to(X0,   (n,) + X0.shape)
    ang = jnp.broadcast_to(ANG0, (n,) + ANG0.shape)
    he  = jnp.broadcast_to(HE0,  (n,) + HE0.shape)
    fc  = jnp.broadcast_to(FACE0,(n,) + FACE0.shape)
    return X, ang, he, fc


# Smoke test
Xb, angb, heb, fcb = reset_envs(4)
keys = jax.random.split(jax.random.PRNGKey(0), 4)
out  = env_step_batch(mgn_params, Xb, angb, heb, fcb, keys)
Xn, angn, hen, fcn, act, lp, val, rew, En = out
print(f'Vectorised env OK — X {Xn.shape}, ang {angn.shape}, rewards {rew.shape}')
print(f'  rewards[:4]={np.array(rew)}')
print(f'  ang changed: {not jnp.allclose(angn[0], angb[0])}  (should be True — angles updated)')


## 4. PPO training (vectorised on GPU)

Full PPO loop in JAX, **256 environments in parallel** via `vmap` + `lax.scan`.

**Components:**
- `rollout`: collects a `MAX_STEPS` trajectory over the `N_ENVS` envs in a single jitted `lax.scan`. `he`/`face` are **excluded** from storage to avoid OOM (topology is near-fixed with small steps).
- `compute_gae`: Generalized Advantage Estimation over `(T, N_ENVS)`, fixed horizon.
- `ppo_update`: clipped PPO loss (surrogate + value + entropy), vmapped over the flattened batch. Uses `HE0`/`FACE0` (fixed topology).
- **Cosine annealing** of the learning rate from `LR_PEAK` to `LR_MIN`.
- **Checkpoint** of the best model (`best_E`) saved to `ppo_mgn_params.pkl`.

**Logs** (every `LOG_EVERY` iters): mean return, mean/min final energy, ΔE%, lr, loss. Automatic stop on NaN/Inf.

In [ ]:
# ── PPO (vectorised, GPU-batched) ─────────────────────────────────────────────
# he/face are NOT stored in the scan trajectory to save GPU memory.
# With ACTION_SCALE=0.003, T1s are extremely rare and he/face barely change,
# so we use HE0/FACE0 for the PPO update (slight approximation, negligible impact).
import pickle
from pathlib import Path

N_ITERS      = 500
LR_PEAK      = 3e-4
LR_MIN       = 1e-6
GAMMA        = 0.99
LAM          = 0.95
CLIP_EPS     = 0.2
N_EPOCHS_PPO = 4
ENT_COEF     = 1e-3
VF_COEF      = 0.5
RET_CLIP     = 50.0
LOG_EVERY    = 10
MODEL_PATH   = Path('/home/mcorbeau/Bureau/vertax/src/vertax/mitotic_spindle/ppo_mgn_params.pkl')

lr_schedule = optax.cosine_decay_schedule(
    init_value=LR_PEAK, decay_steps=N_ITERS * N_EPOCHS_PPO, alpha=LR_MIN / LR_PEAK
)
optimizer_ppo = optax.chain(optax.clip_by_global_norm(0.5), optax.adam(learning_rate=lr_schedule))
mgn_params = init_mgn(jax.random.PRNGKey(0))
opt_state  = optimizer_ppo.init(mgn_params)


def action_log_prob(action, mean, log_std):
    std = jnp.exp(log_std) * ACTION_SCALE
    return -0.5 * jnp.sum(((action - mean) / (std + 1e-8)) ** 2
                          + 2 * jnp.log(std + 1e-8) + jnp.log(2 * jnp.pi))


@jit
def rollout(params, key):
    """Collect MAX_STEPS over N_ENVS. Only X/ang/action/logp/value/reward/E stored
    in scan output — he/face excluded to avoid OOM with 256 envs × 100 steps."""
    X, ang, he, fc = reset_envs(N_ENVS)

    def scan_step(carry, step_key):
        X, ang, he, fc = carry
        keys = jax.random.split(step_key, N_ENVS)
        X2, ang2, he2, fc2, action, logp, value, reward, E_new = env_step_batch(
            params, X, ang, he, fc, keys
        )
        # Store only the lightweight arrays — he/face omitted to save memory
        out = dict(X=X, ang=ang, action=action, logp=logp,
                   value=value, reward=reward, E=E_new)
        return (X2, ang2, he2, fc2), out

    step_keys = jax.random.split(key, MAX_STEPS)
    _, traj = jax.lax.scan(scan_step, (X, ang, he, fc), step_keys)
    return traj   # fields: (MAX_STEPS, N_ENVS, ...)


def compute_gae(rewards, values, gamma=GAMMA, lam=LAM):
    T = rewards.shape[0]
    advs = jnp.zeros_like(rewards)

    def body(t, carry):
        gae_prev, advs = carry
        t_rev    = T - 1 - t
        next_val = jnp.where(t == 0, 0.0, values[jnp.minimum(t_rev + 1, T - 1)])
        delta    = rewards[t_rev] + gamma * next_val - values[t_rev]
        gae_now  = delta + gamma * lam * gae_prev
        advs     = advs.at[t_rev].set(gae_now)
        return (gae_now, advs)

    _, advs = jax.lax.fori_loop(0, T, body, (jnp.zeros(rewards.shape[1]), advs))
    return advs, advs + values


@jit
def ppo_update(params, opt_state, traj, advs, returns):
    """PPO update. Uses HE0/FACE0 (fixed topology) since he/face not in traj."""
    T, N = traj['reward'].shape
    flat = lambda a: a.reshape((T * N,) + a.shape[2:])
    Xf, angf       = flat(traj['X']), flat(traj['ang'])
    actf, logpf    = flat(traj['action']), flat(traj['logp'])
    advf, retf     = flat(advs), flat(returns)
    advf = (advf - advf.mean()) / (advf.std() + 1e-8)
    retf = jnp.clip(retf, -RET_CLIP, RET_CLIP)

    # Tile HE0/FACE0 for the batch
    he_f   = jnp.broadcast_to(HE0,   (T * N,) + HE0.shape)
    face_f = jnp.broadcast_to(FACE0, (T * N,) + FACE0.shape)

    def loss_fn(p):
        def per_sample(X, ang, he, fc, action, old_logp, adv, ret):
            srcs, tgts, tens, deg, mask = build_graph_jax(he, EP0)
            gE = grad_energy_of(X, ang, he, fc)
            mean, log_std, value = mgn_forward_masked(p, X, srcs, tgts, tens, gE, deg, mask)
            logp  = action_log_prob(action, mean, log_std)
            ratio = jnp.exp(jnp.clip(logp - old_logp, -10.0, 10.0))
            pg    = -jnp.minimum(ratio * adv, jnp.clip(ratio, 1-CLIP_EPS, 1+CLIP_EPS) * adv)
            vf    = (value - ret) ** 2
            ent   = jnp.sum(log_std + 0.5 * jnp.log(2 * jnp.pi * jnp.e))
            return pg + VF_COEF * vf - ENT_COEF * ent
        return jnp.mean(vmap(per_sample)(Xf, angf, he_f, face_f, actf, logpf, advf, retf))

    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, opt_state = optimizer_ppo.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    return params, opt_state, loss


# ── Training loop ─────────────────────────────────────────────────────────────
key = jax.random.PRNGKey(0)
iter_returns, iter_energies, iter_losses = [], [], []
best_E = float('inf')
stop_training = False
update_count = 0

print(f'Vectorised PPO — {N_ITERS} iters × {N_ENVS} envs × {MAX_STEPS} steps')
print(f'  ACTION_SCALE={ACTION_SCALE}, target E ≤ {E_bilevel:.2f}')
print(f'{"Iter":>6} {"meanRet":>9} {"meanFinalE":>11} {"minFinalE":>10} {"ΔE%":>7} {"lr":>9} {"loss":>10}')
print('-' * 72)

for it in range(N_ITERS):
    key, rk = jax.random.split(key)
    traj = rollout(mgn_params, rk)
    advs, returns = compute_gae(traj['reward'], traj['value'])

    ep_loss = 0.0
    for _ in range(N_EPOCHS_PPO):
        mgn_params, opt_state, loss = ppo_update(mgn_params, opt_state, traj, advs, returns)
        ep_loss = float(loss)
        update_count += 1
        if not np.isfinite(ep_loss):
            print(f'\n⚠️  NaN/Inf loss at iter {it+1} — stopping.')
            stop_training = True
            break
    if stop_training:
        break

    final_E   = np.array(traj['E'][-1])
    mean_ret  = float(jnp.sum(traj['reward'], axis=0).mean())
    mean_fE   = float(final_E.mean())
    min_fE    = float(final_E.min())
    iter_returns.append(mean_ret)
    iter_energies.append(mean_fE)
    iter_losses.append(ep_loss)

    if min_fE < best_E:
        best_E = min_fE
        with open(MODEL_PATH, 'wb') as f:
            pickle.dump({'params': mgn_params, 'best_E': best_E,
                         'E_bilevel': E_bilevel, 'E_init': E_init, 'iter': it + 1}, f)

    if (it + 1) % LOG_EVERY == 0:
        dpct    = 100 * (E_init - mean_fE) / abs(E_init)
        lr_cur  = float(lr_schedule(update_count))
        reached = '✓' if min_fE <= E_bilevel else ' '
        print(f'{it+1:>6d} {mean_ret:>9.3f} {mean_fE:>11.2f} {min_fE:>10.2f} {dpct:>6.1f}% '
              f'{lr_cur:>9.2e} {ep_loss:>10.4f} {reached}')

if not stop_training:
    with open(MODEL_PATH, 'wb') as f:
        pickle.dump({'params': mgn_params, 'best_E': best_E,
                     'E_bilevel': E_bilevel, 'E_init': E_init, 'iter': N_ITERS}, f)
    print(f'\nModel saved to {MODEL_PATH} (best E = {best_E:.4f})')

print(f'\n{"=" * 45}')
print(f'  Initial   E = {E_init:.4f}')
print(f'  Best PPO  E = {best_E:.4f}  ({100*(E_init-best_E)/abs(E_init):.1f}% reduction)')
print(f'  Bilevel   E = {E_bilevel:.4f}')
print(f'  Gap         = {best_E - E_bilevel:+.4f}')
print(f'{"=" * 45}')


## 5. Evaluation & PPO vs Bilevel comparison

This cell **loads the best saved model** (`ppo_mgn_params.pkl`), rolls out a **deterministic** trajectory (no exploration noise) for `MAX_STEPS`, then compares the result to the bilevel.

**Contents:**
- `eval_step_no_t1`: one deterministic step at fixed topology (no T1), action = policy mean + gradient step on the angles.
- **Diagnostics**: cell areas, `neg_area` (inversion check), final energy.
- **PPO mesh reconstruction** for rendering — `edges_params` is repeated to length `len(edges)` (one entry per half-edge) since that is what `get_plot_mesh` expects.
- **Figure 1**: energy vs step (eval), training curves (smoothed return + energy), comparison bar chart.
- **Figure 2**: mesh geometry — initial | PPO | bilevel, coloured by tension (edges) and area (cells).
- **Summary table**: ΔE of each method vs initial and vs bilevel.

The energy-parameter consistency check (cell 1) guarantees that `E_ppo` and `E_bilevel` use **the same formula and the same parameters**, so the comparison is valid.

In [ ]:
# ── Comparative plots ─────────────────────────────────────────────────────────
import pickle
from pathlib import Path
from vertax import get_plot_mesh, EdgePlot, FacePlot, VertexPlot

MODEL_PATH = Path('/home/mcorbeau/Bureau/vertax/src/vertax/mitotic_spindle/ppo_mgn_params.pkl')

if MODEL_PATH.exists():
    with open(MODEL_PATH, 'rb') as f:
        ckpt = pickle.load(f)
    mgn_params_eval = ckpt['params']
    print(f'Loaded model — best_E={ckpt["best_E"]:.4f}, iter={ckpt["iter"]}')
else:
    mgn_params_eval = mgn_params
    print('No saved model, using in-memory params.')

PLOT_KW = dict(
    edge_plot=EdgePlot.EDGE_PARAMETER, edge_parameters_name='tension',
    face_plot=FacePlot.AREA, face_parameters_name='area',
    vertex_plot=VertexPlot.INVISIBLE,
)

def render_mesh(mesh, title):
    mc = BoundedMesh.copy_mesh(mesh)
    mc.angles = jnp.clip(jnp.array(mc.angles), ANG_MIN, ANG_MAX)
    fig_m, _ = get_plot_mesh(mc, title=title, **PLOT_KW)
    fig_m.canvas.draw()
    w, h = fig_m.canvas.get_width_height()
    img = np.frombuffer(fig_m.canvas.tostring_argb(), dtype=np.uint8).reshape(h, w, 4)
    img = np.roll(img, -1, axis=2)
    plt.close(fig_m)
    return img


# ── Deterministic eval (no T1, fixed topology, angles updated by GD) ──────────
@jit
def eval_step_no_t1(params, X, ang):
    srcs, tgts, tens, deg, mask = build_graph_jax(HE0, EP0)
    gE   = grad_energy_of(X, ang, HE0, FACE0)
    mean, _, _ = mgn_forward_masked(params, X, srcs, tgts, tens, gE, deg, mask)
    X_new, _ = project_action(X, mean, srcs, tgts, mask)
    g_ang    = grad_energy_ang(X_new, ang, HE0, FACE0)
    ang_new  = jnp.clip(ang - ANG_LR * g_ang, ANG_MIN, ANG_MAX)
    E_new    = jnp.nan_to_num(energy_of(X_new, ang_new, HE0, FACE0), nan=1e12, posinf=1e12)
    return X_new, ang_new, E_new


X_e, ang_e = X0, ANG0
ppo_step_energies = [float(energy_of(X_e, ang_e, HE0, FACE0))]
for _ in range(MAX_STEPS):
    X_e, ang_e, E_e = eval_step_no_t1(mgn_params_eval, X_e, ang_e)
    ppo_step_energies.append(float(E_e))
E_ppo_final  = ppo_step_energies[-1]
n_steps_eval = MAX_STEPS

areas_final = np.array(cell_areas(X_e, ang_e, HE0, FACE0))
print(f'Cell areas after {MAX_STEPS} steps:')
for i, a in enumerate(areas_final):
    status = '✓' if a > 0.1 else ('⚠ small' if a > 0 else '✗ negative')
    print(f'  cell {i}: area={a:.4f}  {status}')
print(f'neg_area={float(neg_area(X_e, ang_e, HE0, FACE0)):.4f}, E={E_ppo_final:.2f}')

# ── Reconstruct PPO mesh ──────────────────────────────────────────────────────
# get_plot_mesh iterates over len(edges) half-edges and indexes edges_params[he],
# so edges_params must have one entry PER HALF-EDGE (= len(edges)). The mesh stores
# it per unique edge, so we repeat to match what the plot expects.
mesh_ppo_final = BoundedMesh.copy_mesh(mesh_init)
mesh_ppo_final.vertices = np.array(X_e)
mesh_ppo_final.angles   = np.array(ang_e)

_ep_init = np.asarray(mesh_init.edges_params)
_n_he    = len(np.asarray(mesh_init.edges))
if _ep_init.shape[0] != _n_he:
    mesh_ppo_final.edges_params = np.repeat(_ep_init, _n_he // _ep_init.shape[0])
else:
    mesh_ppo_final.edges_params = _ep_init

# ── Plots ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.plot(ppo_step_energies, color='steelblue', label='PPO (det.)')
ax.axhline(E_bilevel, color='red',  linestyle=':', linewidth=2, label=f'Bilevel ({E_bilevel:.1f})')
ax.axhline(E_init,    color='gray', linestyle=':', linewidth=1, label=f'Initial ({E_init:.1f})')
ax.set_xlabel('Step'); ax.set_ylabel('Energy')
ax.set_title('Energy vs step'); ax.legend(fontsize=9)

ax = axes[1]
window = max(1, min(20, len(iter_energies) // 2))
if len(iter_energies) >= window:
    ret_s = np.convolve(iter_returns,  np.ones(window)/window, mode='valid')
    en_s  = np.convolve(iter_energies, np.ones(window)/window, mode='valid')
else:
    ret_s, en_s = np.array(iter_returns), np.array(iter_energies)
ax2r = ax.twinx()
ax.plot(ret_s, color='steelblue', label='Return')
ax2r.plot(en_s, color='tomato', linestyle='--', label='FinalE')
ax2r.axhline(E_bilevel, color='red', linestyle=':', linewidth=1.5)
ax2r.set_yscale('log')
ax.set_xlabel('Iter'); ax.set_ylabel('Return', color='steelblue')
ax2r.set_ylabel('Energy', color='tomato'); ax.set_title('Training curves')
l1, lab1 = ax.get_legend_handles_labels(); l2, lab2 = ax2r.get_legend_handles_labels()
ax.legend(l1+l2, lab1+lab2, fontsize=9)

ax = axes[2]
methods      = ['Initial', f'PPO\n({n_steps_eval} steps)', 'Bilevel\n(2000 iters)']
energies_bar = [E_init, E_ppo_final, E_bilevel]
bars = ax.bar(methods, energies_bar, color=['gray','steelblue','red'],
              alpha=0.8, edgecolor='black', linewidth=0.8)
for bar, val in zip(bars, energies_bar):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f'{val:.1f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_ylabel('Energy'); ax.set_title('Final energy comparison')

plt.suptitle('PPO vs Bilevel — mesh relaxation', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

fig2, axes2 = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
for ax, (mesh, title) in zip(axes2, [
    (mesh_init,      f'Initial\n(E={E_init:.1f})'),
    (mesh_ppo_final, f'PPO ({n_steps_eval} steps)\n(E={E_ppo_final:.1f})'),
    (mesh_bilevel,   f'Bilevel (2000 iters)\n(E={E_bilevel:.1f})'),
]):
    try:
        ax.imshow(render_mesh(mesh, title))
    except Exception as e:
        ax.text(0.5, 0.5, f'render failed:\n{e}', ha='center', va='center', fontsize=8, wrap=True)
    ax.set_title(title, fontsize=11); ax.axis('off')
fig2.suptitle('Mesh geometry: initial vs PPO vs bilevel', fontsize=13); plt.show()

print(f'\n{"Method":<30} {"Energy":>10} {"ΔE vs initial":>15} {"ΔE vs bilevel":>15}')
print('-' * 72)
for name, E in [('Initial', E_init), (f'PPO ({n_steps_eval} steps)', E_ppo_final),
                ('Bilevel (2000 iters)', E_bilevel)]:
    print(f'{name:<30} {E:>10.2f} {E_init-E:>+15.2f} {E_bilevel-E:>+15.2f}')
